# M08A — Discriminant Functions, Optimisation & SVM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M08-Discriminant-Functions/M08A-Discriminant-Functions-Optimisation-and-SVM.ipynb)

**Pattern Classification practical · approximately 2 hours · M08 · CLO3**

This is the canonical student-facing practical for session `M08A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1A-06-OptimizationandRootFinding.ipynb`, `F1A-07-AlgorithmsforOptimization.ipynb`, `F1A-08-UsingOptimizationRoutines.ipynb`, `F1A-09-SVM.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

Discriminant functions turn parameter estimates into scores; optimisation learns those parameters. You will implement a small gradient-based classifier, then compare linear and nonlinear maximum-margin classifiers.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Express a linear classifier as a discriminant function.
- Optimise logistic loss with gradient descent and inspect convergence.
- Compare linear and kernel SVM decision boundaries.
- Explain the roles of margin, regularisation parameter C, and RBF gamma.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- a gradient-descent loss trace and learned discriminant
- linear and RBF SVM boundary comparisons with numeric metrics
- a cross-validated C/gamma choice explained through margin and locality

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Vectors, dot products, and derivatives
- Binary classification
- Feature scaling

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

A linear discriminant has the form $g(x)=w^Tx+b$ and predicts from the sign of the score. Logistic regression learns $w,b$ by minimising log loss. Gradient descent repeatedly moves opposite the gradient; loss curves help diagnose step size and convergence.

An SVM emphasises margin. `C` controls the penalty for training violations. The RBF kernel allows curved boundaries, while `gamma` controls how local each training point's influence is. Scaling matters because both margin and RBF distance depend on feature units.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles, make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Optimise a linear logistic discriminant

We add a column of ones so the intercept is learned as another weight.

In [ ]:
X_linear, y_linear = make_classification(
    n_samples=500, n_features=2, n_redundant=0, n_informative=2,
    class_sep=1.4, random_state=RANDOM_STATE,
)
X_scaled = StandardScaler().fit_transform(X_linear)
X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled])

def logistic_loss_and_gradient(weights, X, y):
    scores = X @ weights
    loss = np.mean(np.logaddexp(0, scores) - y * scores)
    probabilities = 1 / (1 + np.exp(-np.clip(scores, -35, 35)))
    gradient = X.T @ (probabilities - y) / len(y)
    return loss, gradient

weights = np.zeros(X_design.shape[1])
loss_history = []
learning_rate = 0.25
for _ in range(400):
    loss, gradient = logistic_loss_and_gradient(weights, X_design, y_linear)
    loss_history.append(loss)
    weights -= learning_rate * gradient

linear_predictions = (X_design @ weights >= 0).astype(int)
print("weights:", weights.round(3))
print(f"training accuracy: {accuracy_score(y_linear, linear_predictions):.3f}")

### 2. Inspect optimisation evidence

A decreasing curve is necessary but not sufficient: also check scale, stopping, and held-out performance.

In [ ]:
plt.plot(loss_history)
plt.xlabel("gradient step")
plt.ylabel("logistic loss")
plt.title("Optimisation trace")
plt.show()
print(f"initial loss={loss_history[0]:.4f}, final loss={loss_history[-1]:.4f}")

### 3. Compare linear and RBF SVMs on nonlinear data

Both pipelines learn scaling only from the training split.

In [ ]:
X, y = make_circles(n_samples=650, factor=0.45, noise=0.13, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

def svm_pipeline(kernel, **kwargs):
    return Pipeline([
        ("scale", StandardScaler()),
        ("svm", SVC(kernel=kernel, **kwargs)),
    ])

linear_svm = svm_pipeline("linear", C=1.0).fit(X_train, y_train)
rbf_svm = svm_pipeline("rbf", C=1.0, gamma="scale").fit(X_train, y_train)
print(f"linear SVM accuracy: {accuracy_score(y_test, linear_svm.predict(X_test)):.3f}")
print(f"RBF SVM accuracy: {accuracy_score(y_test, rbf_svm.predict(X_test)):.3f}")

### 4. Visualise the discriminant boundaries

The zero contour is where the SVM decision function changes sign.

In [ ]:
def plot_svm(ax, model, title):
    x0 = np.linspace(-1.4, 1.4, 240)
    x1 = np.linspace(-1.4, 1.4, 240)
    xx, yy = np.meshgrid(x0, x1)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    scores = model.decision_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, scores > 0, alpha=0.22, cmap="coolwarm")
    ax.contour(xx, yy, scores, levels=[-1, 0, 1], colors="black", linestyles=["--", "-", "--"])
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=18, edgecolor="k", linewidth=0.2)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_svm(axes[0], linear_svm, "Linear SVM")
plot_svm(axes[1], rbf_svm, "RBF SVM")
plt.tight_layout()
plt.show()

## Student Challenge

Tune the RBF SVM over `C=[0.1, 1, 10, 100]` and `gamma=[0.1, 1, 10]` using five-fold stratified cross-validation on the training split. Plot or tabulate the validation surface, then explain the selected boundary in terms of margin violations and locality.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

parameter_grid = {"svm__C": [0.1, 1, 10, 100], "svm__gamma": [0.1, 1, 10]}
# TODO: configure GridSearchCV around an RBF `svm_pipeline`, fit on X_train/y_train,
# inspect cv_results_, and evaluate the selected estimator once on X_test/y_test.
challenge_search = None

### Challenge success criteria

- Scaling occurs within every fold.
- The selection metric and fold design are stated.
- The held-out set is used once after selection.
- The interpretation connects C and gamma to the observed boundary.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise invalid-hyperparameter and missing-input behaviour.
try:
    svm_pipeline("rbf", C=0).fit(X_train, y_train)
except ValueError as error:
    print("Expected invalid-C failure:", str(error).split("\n")[0])
else:
    raise AssertionError("SVC should reject C=0.")

try:
    rbf_svm.predict(np.array([[np.nan, 0.0]]))
except ValueError as error:
    print("Expected missing-value failure:", str(error).split("\n")[0])
else:
    raise AssertionError("The SVM workflow should reject NaN input.")

In [ ]:
assert np.isfinite(loss_history).all()
assert loss_history[-1] < loss_history[0] * 0.8
assert weights.shape == (3,)
assert len(rbf_svm.named_steps["svm"].support_) > 0
assert accuracy_score(y_test, rbf_svm.predict(X_test)) > 0.9
print("M08A guided-practice checks passed.")

## Extension Task

Add label noise with `flip_y` in a generated two-class dataset. Track how the best C changes and relate the result to soft-margin regularisation.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Sources: `F1A-06`, `F1A-07`, `F1A-08`, and `F1A-09-SVM.ipynb`.
- Cortes and Vapnik (1995), “Support-Vector Networks.”
- scikit-learn User Guide: [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)